# SYNTHETIC TECHNICAL VALIDATION — NOT BAF RESULTS
960 generated rows; reduced budgets; two CPU seeds. This checks execution and evidence, not research performance.

# 14 — Latent Score-Based Reweighting

Import trực tiếp VAE/EDM/diffusion class tác giả; chuyển dữ liệu sang BAF và giới hạn ngân sách30/50 epoch. LSR dùng latent MLP cùng baseline none/balanced; có thêm transfer sang LightGBM với baseline riêng. Không gọi đây là tái lập nguyên xi các con số ICML2025. Nếu bỏ LSR từ11, notebook này in SKIPPED.

Không sửa cấu hình ở notebook này. Muốn đổi cấu hình hãy tạo campaign mới ở11.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "baf.py").is_file():
    raise RuntimeError("Mở notebook từ thư mục BAF_project hoặc BAF_project/notebooks.")

CACHE_ROOT = ROOT.parent.parent / ".cache" / "baf-research"
if os.name == "nt":
    CACHE_ROOT = Path(r"D:\UIT\.cache\baf-research")
for name, subdirectory in {
    "PIP_CACHE_DIR": "pip",
    "TORCH_HOME": "torch",
    "XDG_CACHE_HOME": "misc",
    "TEMP": "temp",
    "TMP": "temp",
}.items():
    folder = CACHE_ROOT / subdirectory
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(folder)

sys.path.insert(0, str(ROOT / "src"))
import gru

print("Python:", sys.executable)
print("Project:", ROOT)
print("Module:", gru.RESEARCH_VERSION)


Python: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python3
Project: /workspace/scratch/a3538503433e/work/research11/synthetic_validation
Module: baf-research-20261028-v1


In [2]:
# Điền đường dẫn in ở notebook 11 nếu đã có nhiều campaign.
# Ví dụ: Path(r"D:\UIT\DS317\BAF_project\results\research_campaigns\weighting_research_...")
CAMPAIGN_DIR = None
CAMPAIGN = gru.research_resolve_campaign(ROOT, CAMPAIGN_DIR)
PLAN = gru.research_read(CAMPAIGN / "campaign.json")

print("Campaign:", CAMPAIGN)
print("Mode:", PLAN["mode"])
print("Suites:", PLAN["suites"])
print("Seeds:", PLAN["config"]["seeds"])


Campaign: /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05
Mode: synthetic
Suites: ['losses', 'meta', 'lsr', 'xgboost']
Seeds: [42, 317]


## 1. Chạy training có notebook output thật

        Cell khởi chạy một Python process mới bằng đúng interpreter của kernel. Process thực thi từng cell trong notebook con, lưu stdout/display/error sau từng cell và in log trực tiếp bên dưới. Fit hoàn tất được kiểm hash rồi reuse khi resume. Nếu bị ngắt, không xóa campaign. Cell này có thể chạy lâu.

In [3]:
SUITE = 'lsr'
RUN_DIR = None
if SUITE in PLAN["suites"]:
    RUN_DIR = gru.research_launch_suite(CAMPAIGN, SUITE)
else:
    print("SKIPPED: suite không thuộc phạm vi đã chọn ở11:", SUITE)


BEGIN_STAGE 1
END_STAGE 1
BEGIN_STAGE 2
RESEARCH_STUDY lsr mode= synthetic device= cpu
END_STAGE 2
BEGIN_STAGE 3
self.category_embeddings.weight.shape=torch.Size([20, 4])
LSR_VAE_EPOCH {'epoch': 1, 'train_loss': 2.7424113402205905, 'meta_reconstruction_ce': 1.5218091011047363, 'beta': 0.01}
LSR_VAE_EPOCH {'epoch': 2, 'train_loss': 2.310705602838752, 'meta_reconstruction_ce': 1.3493363857269287, 'beta': 0.01}
self.category_embeddings.weight.shape=torch.Size([20, 4])
LSR_DIFFUSION_EPOCH {'epoch': 1, 'class': 0, 'train_edm_loss': 0.6695556797479328}
LSR_DIFFUSION_EPOCH {'epoch': 2, 'class': 0, 'train_edm_loss': 0.7128398794876901}
LSR_DIFFUSION_EPOCH {'epoch': 1, 'class': 1, 'train_edm_loss': 0.7235649044697101}
LSR_DIFFUSION_EPOCH {'epoch': 2, 'class': 1, 'train_edm_loss': 0.6565154011432941}
LSR_SCORE_BATCH 0 128 / 712
LSR_SCORE_BATCH 0 256 / 712
LSR_SCORE_BATCH 0 384 / 712
LSR_SCORE_BATCH 0 512 / 712
LSR_SCORE_BATCH 0 640 / 712
LSR_SCORE_BATCH 0 712 / 712
LSR_SCORE_BATCH 1 128 / 712
LS

## 2. Xem kết quả validation đã khóa

        Chưa có test metric ở notebook này. Model, configuration và threshold được lưu trước khi mở test ở16.

In [4]:
if RUN_DIR is not None:
    display(gru.research_suite_overview(RUN_DIR))
    print("Notebook training thật:")
    for executed in sorted((RUN_DIR / "attempts").glob("*/training_executed.ipynb")):
        print(executed)


Notebook training thật:
/workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/research_campaigns/weighting_research_20261028_142640_1f37cc05/suites/lsr/attempts/20261028_142702_50f73bbe/training_executed.ipynb


,method,seed,validation_ap,threshold,fit_seconds,family,feature
0,lsr_lgbm_balanced,42,0.310582,0.562963,0.046774,lsr_lgbm,dense
1,lsr_lgbm_balanced,317,0.351956,0.537464,0.040756,lsr_lgbm,dense
2,lsr_lgbm_none,42,0.278601,0.168671,0.122297,lsr_lgbm,dense
3,lsr_lgbm_none,317,0.310787,inf,0.029818,lsr_lgbm,dense
4,lsr_lgbm_weighted,42,0.279374,0.168907,0.038997,lsr_lgbm,dense
5,lsr_lgbm_weighted,317,0.310787,inf,0.032870,lsr_lgbm,dense
6,lsr_nn_balanced,42,0.233689,0.651888,0.777485,lsr_nn,latent
7,lsr_nn_balanced,317,0.371888,0.494705,0.806523,lsr_nn,latent
8,lsr_nn_none,42,0.235810,0.156202,0.739685,lsr_nn,latent
9,lsr_nn_none,317,0.251543,inf,0.728817,lsr_nn,latent


## 3. Xuất proof trong một lượt

        Không cần Ctrl+S hoặc chạy cell cuối hai lần. Full proof giữ trên máy; review nhỏ hơn để gửi, ghi rõ phần đã bỏ. Notebook `verify.ipynb` được execute thật, kiểm hash/score và không fit lại.

In [5]:
if RUN_DIR is not None:
    PROOF = gru.research_export_proof(RUN_DIR, ROOT)
    for kind, path in PROOF.items():
        print(kind, "→", path)


RESEARCH_FROZEN_SUITE_VERIFIED 12
FULL_PROOF /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/lsr_53f817ea_full_proof.zip
PARTIAL_REVIEW /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/lsr_53f817ea_review.zip
full_proof → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/lsr_53f817ea_full_proof.zip
review → /workspace/scratch/a3538503433e/work/research11/synthetic_validation/results/proofs/lsr_53f817ea_review.zip
